<a href="https://colab.research.google.com/github/jazaineam1/BigData2026/blob/main/Cuadernos/Dask.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dask para una sesión de computación distribuida

<img src="https://docs.dask.org/en/latest/_images/dask_horizontal.svg" align="right" width="32%" alt="Logo de Dask">

Este cuaderno está diseñado para enseñar **Dask desde cero** con una narrativa pensada para estudiantes que todavía no dominan la computación distribuida.


- En **Colab** usamos Dask para entender conceptos, la API y ejemplos pequeños.
- En tu **infraestructura Docker local** mostramos el valor real del clúster con `scheduler`, `workers`, dashboard y benchmark contra Pandas.
- El objetivo no es decir que Dask siempre gana, sino entender **cuándo sí tiene sentido**.

## Objetivos de aprendizaje

Al finalizar la sesión, la persona debería poder:

1. Explicar qué es Dask y cómo se relaciona con Pandas.
2. Entender primero un flujo secuencial en Pandas antes de paralelizarlo.
3. Entender la diferencia entre `client`, `scheduler`, `workers` y `particiones`.
4. Reconocer que Dask evalúa de forma perezosa (`lazy`).
5. Comparar un flujo simple en Pandas vs Dask.
6. Ejecutar un benchmark real sobre el clúster Docker del curso.


## estructura de nodos y distribución

Esta imagen es útil para explicar la intuición del clúster antes de entrar al código.

<img src="https://raw.githubusercontent.com/jazaineam1/BigData2026/refs/heads/main/Images/Cluster/nodos.png" alt="Estructura de nodos" width="760"/>

## Instalacion por defecto de la API de Dask

In [1]:
# Descomenta esta celda y ejecútala UNA SOLA VEZ
%pip install --quiet --no-cache-dir \
    "dask[distributed,dataframe]==2024.5.1" \
    "dask-expr==1.1.1" \
    "distributed==2024.5.1" \
    pandas \
    numpy \
    pyarrow \
    graphviz

Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
warnings.filterwarnings("ignore")

import os
import sys
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd

# Importar Dask correctamente
try:
    import dask
    import dask.dataframe as dd
    from dask.distributed import Client, wait
except ImportError as e:
    print(f"❌ Error importando Dask: {e}")
    print("💡 Ejecuta la celda anterior de instalación primero")
    sys.exit(1)

from IPython.display import display

# Detectar Colab
try:
    from google.colab import files  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# Configurar Dask para Colab
if IN_COLAB:
    # En Colab, usar scheduler de threads (no multiprocessing)
    dask.config.set(scheduler='threads')
    print("🔵 Ejecutando en Google Colab")
else:
    dask.config.set(scheduler='synchronous')
    print("🖥️ Ejecutando localmente")

# Configurar Pandas
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

print(f"✅ Dask {dask.__version__} listo para usar")
print(f"✅ Pandas {pd.__version__}")
print(f"✅ NumPy {np.__version__}")

🖥️ Ejecutando localmente
✅ Dask 2024.5.1 listo para usar
✅ Pandas 3.0.2
✅ NumPy 2.4.4


## 1. ¿qué problema resuelve Dask?

Pandas funciona muy bien cuando:

- los datos caben cómodamente en memoria,
- el trabajo cabe en una sola máquina,
- y el costo de coordinar un sistema distribuido sería mayor que el beneficio.

Dask entra en escena cuando queremos:

- dividir datos en **particiones**,
- procesarlas en paralelo,
- trabajar con un API muy parecida a Pandas,
- o conectarnos a un clúster con varios workers.

### Mapa mental simple

- **Client**: desde donde tú envías el trabajo.
- **Scheduler**: quien coordina qué tarea se ejecuta y en qué worker.
- **Worker**: proceso que hace cómputo y usa memoria.
- **Partition**: fragmento del dataset; en Dask DataFrame cada partición se parece a un DataFrame pequeño de Pandas.

## Imagen conceptual de Dask DataFrame

Esta imagen ayuda a explicar que un Dask DataFrame es una colección de múltiples particiones, no un único bloque monolítico en memoria.

<img src="https://docs.dask.org/en/stable/_images/dask-dataframe.svg" align="center" width="780" alt="Dask DataFrame">

## 2. primero Pandas


Primero queremos que el estudiante vea un flujo normal:

- leer datos,
- agrupar,
- calcular una métrica,
- interpretar el resultado.

Después sí mostramos cómo Dask toma esa misma idea y la ejecuta por particiones.

In [3]:
ventas_secuencial = pd.DataFrame(
    {
        "region": ["Norte", "Sur", "Norte", "Centro", "Sur", "Occidente"],
        "ventas": [120, 80, 200, 150, 90, 300],
        "unidades": [2, 1, 4, 3, 1, 5],
    }
)

ventas_por_region_pd = ventas_secuencial.groupby("region")[["ventas", "unidades"]].sum()

print("DataFrame original en Pandas")
display(ventas_secuencial)

print("Resultado secuencial con Pandas")
display(ventas_por_region_pd)

DataFrame original en Pandas


,region,ventas,unidades
0,Norte,120,2
1,Sur,80,1
2,Norte,200,4
3,Centro,150,3
4,Sur,90,1
5,Occidente,300,5


Resultado secuencial con Pandas


,ventas,unidades
region,,
Centro,150,3
Norte,320,6
Occidente,300,5
Sur,170,2


### Aquí todavía no hay clúster, ni workers, ni scheduler.

Solo hay una idea sencilla: **tomar datos tabulares y aplicar transformaciones**.
Esa misma idea la vamos a conservar cuando pasemos a Dask.

## 3. La misma idea, ahora con Dask DataFrame

Ahora sí repetimos la misma lógica, pero con Dask. Así el estudiante compara:

- misma intención analítica,
- sintaxis parecida,
- ejecución distinta.

## 4. Primer contacto con `dask.delayed`

En Dask, `delayed` permite convertir código Python normal en un **grafo de tareas diferidas (lazy)** que solo se ejecuta cuando se solicita explícitamente.

---

### Idea clave

Al usar `dask.delayed`:

* No se ejecuta el código inmediatamente
* Se construye un grafo de tareas (task graph)
* La ejecución ocurre únicamente al llamar `.compute()`

---

### Funcionamiento básico

```python
from dask import delayed

@delayed
def suma(a, b):
    return a + b

resultado = suma(1, 2)
```

En este punto, `resultado` no es `3`, sino una **tarea pendiente dentro de un grafo**.

---

### Construcción del grafo

Las operaciones pueden encadenarse:

```python
@delayed
def multiplica(a, b):
    return a * b

x = suma(1, 2)
y = multiplica(x, 10)
```

Aquí se define un flujo lógico de dependencias, pero **no se ejecuta nada aún**.

---

### Ejecución

```python
y.compute()
```

Al llamar `.compute()`, Dask:

* Analiza el grafo completo
* Optimiza el orden de ejecución
* Ejecuta tareas en paralelo cuando es posible
* Devuelve el resultado final

---

```python
from dask import delayed
import time

@delayed
def carga_datos(x):
    time.sleep(1)
    return x

@delayed
def procesa(x):
    return x * 2

datos = [carga_datos(i) for i in range(5)]
resultados = [procesa(d) for d in datos]

total = delayed(sum)(resultados)

print(total.compute())
```

Aunque el código parece secuencial, Dask puede ejecutar las cargas en paralelo.

---

### Diferencia con Pandas

* Pandas ejecuta las operaciones inmediatamente
* Dask construye un plan de ejecución y lo evalúa al final

---

### Cuándo usar `delayed`

Es útil cuando:

* Trabajas con funciones personalizadas
* No utilizas DataFrames directamente
* Necesitas paralelizar código Python general
* Requieres control explícito del flujo de ejecución

---


In [4]:
from dask import delayed

@delayed
def suma(a, b):
    return a + b

resultado = suma(1, 2)

In [5]:
@delayed
def multiplica(a, b):
    return a * b

x = suma(1, 2)
y = multiplica(x, 10)

In [6]:
y

Delayed('multiplica-c01b7542-eda1-48fd-9209-92ee79c0d1a7')

In [6]:
y.compute()

30

 hasta que no llamamos `compute()`, Dask solo está construyendo el plan de ejecución.

##  `.compute()`

* Ejecuta todas las operaciones pendientes
* Devuelve el resultado final en memoria (como Pandas)
* Es obligatorio para ver resultados reales

**Ejemplo:**

```python
resultado = df["col1"].mean()

print(resultado)            # no ejecuta
print(resultado.compute())  # ejecuta
```

**Idea clave:**
Dask no ejecuta nada hasta que llamas `.compute()`

---

## `.persist()`

* Ejecuta las operaciones
* Guarda el resultado en memoria (como cache)
* Devuelve un DataFrame de Dask optimizado
* Sirve para reutilizar datos sin recalcular

**Ejemplo:**

```python
df_filtrado = df[df["col1"] > 5].persist()

df_filtrado.mean().compute()
df_filtrado.sum().compute()
```

---

##  Diferencia principal

* `.compute()` → ejecuta y devuelve resultado final
* `.persist()` → ejecuta y deja datos listos para seguir trabajando

---

##  Cuándo usar cada uno

**Usa `.compute()`**

* Cuando necesitas el resultado final
* Para imprimir o exportar

**Usa `.persist()`**

* Cuando vas a reutilizar el mismo DataFrame
* Después de operaciones costosas (filtros, joins)

---

##  Error típico

```python
df[df["col1"] > 5].mean().compute()
df[df["col1"] > 5].sum().compute()
```

Se recalcula todo dos veces.

**Mejor:**

```python
df_filtrado = df[df["col1"] > 5].persist()

df_filtrado.mean().compute()
df_filtrado.sum().compute()
```

## 5. Levantar un clúster local pequeño dentro del notebook

En Colab y en una sesión local sencilla podemos usar un clúster pequeño en la misma máquina solo para ilustrar conceptos.

Esto **no reemplaza** el clúster Docker del curso; solo nos ayuda a enseñar la API.

In [7]:
try:
    client.close()
except Exception:
    pass

client = Client(
    processes=not IN_COLAB,
    n_workers=2,
    threads_per_worker=1,
    memory_limit="1GB",
)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 2
Total threads: 2,Total memory: 1.86 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:50264,Workers: 2
Dashboard: http://127.0.0.1:8787/status,Total threads: 2
Started: Just now,Total memory: 1.86 GiB
Comm: tcp://127.0.0.1:50278,Total threads: 1
Dashboard: http://127.0.0.1:50279/status,Memory: 0.93 GiB
Nanny: tcp://127.0.0.1:50267,


2026-04-21 23:20:04,453 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 012b8594c112d558d0076749218fec50 initialized by task ('hash-join-transfer-012b8594c112d558d0076749218fec50', 1) executed on worker tcp://127.0.0.1:50275
2026-04-21 23:20:04,814 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 012b8594c112d558d0076749218fec50 deactivated due to stimulus 'task-finished-1776831604.8122563'


In [8]:
print("Dashboard:", client.dashboard_link)

Dashboard: http://127.0.0.1:8787/status


## 3. La misma idea con Dask DataFrame: similar a Pandas, pero por particiones

In [9]:
pdf_demo = pd.DataFrame(
    {
        "region": ["Norte", "Sur", "Norte", "Centro", "Sur", "Occidente"],
        "ventas": [120, 80, 200, 150, 90, 300],
        "unidades": [2, 1, 4, 3, 1, 5],
    }
)

ddf_demo = dd.from_pandas(pdf_demo, npartitions=3)

print("Tipo en Pandas:", type(pdf_demo))
print("Tipo en Dask:", type(ddf_demo))
print("Particiones:", ddf_demo.npartitions)

ddf_demo

Tipo en Pandas: <class 'pandas.DataFrame'>
Tipo en Dask: <class 'dask_expr._collection.DataFrame'>
Particiones: 3


,region,ventas,unidades
npartitions=3,,,
0,string,int64,int64
2,...,...,...
4,...,...,...
5,...,...,...


In [10]:
print("Primera partición como Pandas:")
display(ddf_demo.partitions[0].compute())

print("Agregación perezosa:")
agg_demo = ddf_demo.groupby("region")["ventas"].sum()
print(agg_demo)

print("Resultado materializado:")
display(agg_demo.compute())

Primera partición como Pandas:


,region,ventas,unidades
0,Norte,120,2
1,Sur,80,1


Agregación perezosa:
Dask Series Structure:
npartitions=1
    int64
      ...
Dask Name: sum, 3 expressions
Expr=Sum(frame=df[['region', 'ventas']], observed=True, chunk_kwargs={'numeric_only': False}, aggregate_kwargs={'numeric_only': False}, _slice='ventas')
Resultado materializado:


region
Norte        320
Sur          170
Centro       150
Occidente    300
Name: ventas, dtype: int64


##  `.compute()`

* Ejecuta todas las operaciones pendientes
* Devuelve el resultado final en memoria (como Pandas)
* Es obligatorio para ver resultados reales

**Ejemplo:**

```python
resultado = df["col1"].mean()

print(resultado)            # no ejecuta
print(resultado.compute())  # ejecuta
```

**Idea clave:**
Dask no ejecuta nada hasta que llamas `.compute()`

---

## `.persist()`

* Ejecuta las operaciones
* Guarda el resultado en memoria (como cache)
* Devuelve un DataFrame de Dask optimizado
* Sirve para reutilizar datos sin recalcular

**Ejemplo:**

```python
df_filtrado = df[df["col1"] > 5].persist()

df_filtrado.mean().compute()
df_filtrado.sum().compute()
```

---

##  Diferencia principal

* `.compute()` → ejecuta y devuelve resultado final
* `.persist()` → ejecuta y deja datos listos para seguir trabajando

---

##  Cuándo usar cada uno

**Usa `.compute()`**

* Cuando necesitas el resultado final
* Para imprimir o exportar

**Usa `.persist()`**

* Cuando vas a reutilizar el mismo DataFrame
* Después de operaciones costosas (filtros, joins)

---

##  Error típico

```python
df[df["col1"] > 5].mean().compute()
df[df["col1"] > 5].sum().compute()
```

Se recalcula todo dos veces.

**Mejor:**

```python
df_filtrado = df[df["col1"] > 5].persist()

df_filtrado.mean().compute()
df_filtrado.sum().compute()
```






Dask **no es magia**. Tiene costo de coordinación.

Por eso, con datasets pequeños, muchas veces:

- Pandas es más simple,
- Pandas es más rápido,
- y Dask solo añade overhead.

Eso es normal y es parte de lo que queremos enseñar.

 **tabla práctica con las funciones y patrones más usados**, indicando si la sintaxis es igual o cambia.

---

## 🧠 Pandas vs Dask — Tabla comparativa de funciones clave

| Categoría   | Operación | Pandas              | Dask                   | ¿Igual?      |
| ----------- | --------- | ------------------- | ---------------------- | ------------ |
| **Lectura** | CSV       | `pd.read_csv()`     | `dd.read_csv()`        | ⚠️ Similar   |
|             | Parquet   | `pd.read_parquet()` | `dd.read_parquet()`    | ✅ Igual      |
|             | Excel     | `pd.read_excel()`   | ❌ No soportado directo | ❌ Diferente  |
|             | SQL       | `pd.read_sql()`     | `dd.read_sql_table()`  | ⚠️ Diferente |
|             | JSON      | `pd.read_json()`    | `dd.read_json()`       | ⚠️ Limitado  |

---

| **Exploración** | | | |
|---|---|---|---|
| Head | `df.head()` | `df.head()` | ✅ Igual |
| | Tail | `df.tail()` | ⚠️ Limitado | ❌ Diferente |
| | Info | `df.info()` | ❌ No existe | ❌ Diferente |
| | Describe | `df.describe()` | `df.describe()` | ⚠️ Parcial |

---

| **Selección** || | |
|---|---|---|---|
| Columnas | `df["col"]` | `df["col"]` | ✅ Igual |
| | Filtrado | `df[df.col > 5]` | igual | ✅ Igual |
| | loc | `df.loc[]` | ⚠️ limitado | ❌ Diferente |
| | iloc | `df.iloc[]` | ❌ no soportado | ❌ Diferente |

---

| **Transformación** | | | |
|---|---|---|---|
|Assign | `df.assign()` | `df.assign()` | ✅ Igual |
| | Rename | `df.rename()` | `df.rename()` | ✅ Igual |
| | Drop | `df.drop()` | `df.drop()` | ✅ Igual |
| | Sort | `df.sort_values()` | ⚠️ costoso | ⚠️ Similar |

---

| **Agregación** | | | |
|---|---|---|---|
|Mean | `df.mean()` | `df.mean()` + `.compute()` | ⚠️ Diferente |
| | Sum | `df.sum()` | igual + `.compute()` | ⚠️ Diferente |
| | Count | `df.count()` | igual + `.compute()` | ⚠️ Diferente |
| | Groupby | `df.groupby().agg()` | igual | ⚠️ Similar |

---

| **Merge / Join** | | | |
|---|---|---|---|
|Merge | `pd.merge()` | `dd.merge()` | ⚠️ Similar |
| | Join | `df.join()` | `df.join()` | ⚠️ Limitado |

---

| **Apply / UDF** | | | |
|---|---|---|---|
|Apply | `df.apply()` | `df.apply(meta=...)` | ❌ Diferente |
| | Map | `df.map()` | ⚠️ limitado | ❌ Diferente |

---

| **Missing values** | | | |
|---|---|---|---|
|isnull | `df.isnull()` | `df.isnull()` | ✅ Igual |
| | fillna | `df.fillna()` | `df.fillna()` | ✅ Igual |
| | dropna | `df.dropna()` | `df.dropna()` | ✅ Igual |

---

| **Output** || | |
|---|---|---|---|
| to_csv | `df.to_csv()` | `df.to_csv()` | ⚠️ Diferente (varios archivos) |
| | to_parquet | `df.to_parquet()` | `df.to_parquet()` | ✅ Igual |

---

| **Ejecución 🔥** | | ||
|---|---|---|---|
| Evaluación | Inmediata | Lazy (perezosa) | ❌ Diferente |
| | Ejecutar | automático | `.compute()` | ❌ Clave |
| | Persistir | ❌ | `.persist()` | ❌ Dask only |

---


* **80% de la sintaxis es igual**
* **Dask necesita `.compute()` para ejecutar**

---


In [11]:
import pandas as pd
import dask.dataframe as dd
import numpy as np

# -----------------------------
# 1. Crear datos simulados
# -----------------------------
np.random.seed(42)

pdf = pd.DataFrame({
    "col1": np.arange(1, 11),
    "col2": np.random.randint(10, 100, 10),
    "grupo": np.random.choice(["A", "B"], 10)
})

# Convertir a Dask (2 particiones)
df = dd.from_pandas(pdf, npartitions=2)

print("DataFrame original:")
print(df.compute())

# -----------------------------
# 2. Exploración
# -----------------------------
print("\nHead:")
print(df.head())

print("\nDescribe:")
print(df.describe().compute())

# -----------------------------
# 3. Selección y filtrado
# -----------------------------
print("\nSeleccionar columna:")
print(df["col1"].compute())

print("\nFiltrar col1 > 5:")
print(df[df["col1"] > 5].compute())

# -----------------------------
# 4. Transformaciones
# -----------------------------
df2 = df.assign(col3=df["col1"] * 2)
df2 = df2.rename(columns={"col1": "nueva_col1"})
df2 = df2.drop(columns=["col2"])

print("\nTransformaciones:")
print(df2.compute())

# -----------------------------
# 5. Agregaciones
# -----------------------------
print("\nMedia col1:")
print(df["col1"].mean().compute())

print("\nGroupby:")
print(df.groupby("grupo")["col1"].mean().compute())

# -----------------------------
# 6. Merge
# -----------------------------
df_merge = dd.merge(df, df, on="col1", suffixes=("_left", "_right"))

print("\nMerge:")
print(df_merge.compute())

# -----------------------------
# 7. Apply con meta
# -----------------------------
def duplicar(x):
    return x * 2

df["duplicado"] = df["col1"].apply(duplicar, meta=('col1', 'int64'))

print("\nApply:")
print(df.compute())

# -----------------------------
# 8. Valores nulos
# -----------------------------
df_null = df.copy()
df_null["col2"] = df_null["col2"].mask(df_null["col2"] > 50)

print("\nCon nulos:")
print(df_null.compute())

print("\nFillna:")
print(df_null.fillna(0).compute())

print("\nDropna:")
print(df_null.dropna().compute())

# -----------------------------
# 9. Ordenamiento
# -----------------------------
print("\nSort:")
print(df.sort_values("col1").compute())

# -----------------------------
# 10. Persistencia
# -----------------------------
df_persist = df.persist()

print("\nPersist ejecutado (no imprime nada, pero optimiza)")

DataFrame original:
   col1  col2 grupo
0     1    61     A
1     2    24     B
2     3    81     B
3     4    70     B
4     5    30     A
5     6    92     B
6     7    96     A
7     8    84     B
8     9    84     B
9    10    97     B

Head:
   col1  col2 grupo
0     1    61     A
1     2    24     B
2     3    81     B
3     4    70     B
4     5    30     A

Describe:
           col1       col2
count  10.00000  10.000000
mean    5.50000  71.900000
std     3.02765  26.168047
min     1.00000  24.000000
25%     3.00000  61.000000
50%     5.00000  81.000000
75%     8.00000  92.000000
max    10.00000  97.000000

Seleccionar columna:
0     1
1     2
2     3
3     4
4     5
5     6
6     7
7     8
8     9
9    10
Name: col1, dtype: int64

Filtrar col1 > 5:
   col1  col2 grupo
5     6    92     B
6     7    96     A
7     8    84     B
8     9    84     B
9    10    97     B

Transformaciones:
   nueva_col1 grupo  col3
0           1     A     2
1           2     B     4
2           3   

## 6. Crear un dataset de ventas para comparar Pandas vs Dask

Aquí construiremos un dataset moderado, dividido en varios CSV. Esto nos permite simular el patrón típico de Dask: **muchos archivos, varias particiones y operaciones repetidas**.

In [14]:
# 6. Generar datos sintéticos para demostración en Colab

import numpy as np
import pandas as pd
import tempfile
from pathlib import Path

# Crear directorio temporal para datos
DATA_DIR = Path(tempfile.gettempdir()) / "dask_demo"
DATA_DIR.mkdir(exist_ok=True)

# Generar múltiples CSV con datos sintéticos
np.random.seed(42)

n_particiones = 6
filas_por_particion = 50000

archivos = []

print("Generando datos sintéticos...")
for i in range(n_particiones):
    df_part = pd.DataFrame({
        "fecha": pd.date_range("2024-01-01", periods=filas_por_particion, freq="h"),
        "region": np.random.choice(
            ["Norte", "Sur", "Centro", "Occidente", "Oriental"],
            filas_por_particion
        ),
        "categoria": np.random.choice(["A", "B", "C", "D"], filas_por_particion),
        "ventas": np.random.uniform(100, 5000, filas_por_particion),
        "unidades": np.random.randint(1, 100, filas_por_particion),
        "cliente_id": np.random.randint(1000, 5000, filas_por_particion),
    })

    archivo = DATA_DIR / f"ventas_{i:02d}.csv"
    df_part.to_csv(archivo, index=False)
    archivos.append(archivo)
    print(f"  ✅ Creado: {archivo.name} ({len(df_part):,} filas)")

print(f"\n📁 Datos en: {DATA_DIR}")
print(f"📊 Total de archivos: {len(archivos)}")
print(f"📈 Filas totales estimadas: {filas_por_particion * n_particiones:,}")

Generando datos sintéticos...
  ✅ Creado: ventas_00.csv (50,000 filas)
  ✅ Creado: ventas_01.csv (50,000 filas)
  ✅ Creado: ventas_02.csv (50,000 filas)
  ✅ Creado: ventas_03.csv (50,000 filas)
  ✅ Creado: ventas_04.csv (50,000 filas)
  ✅ Creado: ventas_05.csv (50,000 filas)

📁 Datos en: C:\Users\ser_s\AppData\Local\Temp\dask_demo
📊 Total de archivos: 6
📈 Filas totales estimadas: 300,000


In [15]:
pdf = pd.concat((pd.read_csv(f) for f in archivos), ignore_index=True)
ddf = dd.read_csv(str(DATA_DIR / "ventas_*.csv"), blocksize=None)

print("Filas en Pandas:", f"{len(pdf):,}")
print("Particiones en Dask:", ddf.npartitions)

ddf.head()

Filas en Pandas: 300,000
Particiones en Dask: 6


,fecha,region,categoria,ventas,unidades,cliente_id
0,2024-01-01 00:00:00,Occidente,C,4775.480660,78,2437
1,2024-01-01 01:00:00,Oriental,C,4586.643633,49,1693
2,2024-01-01 02:00:00,Centro,C,2776.052748,68,1467
3,2024-01-01 03:00:00,Oriental,D,1107.795658,57,2728
4,2024-01-01 04:00:00,Oriental,C,4564.577989,39,4724


## 6. Mismo pipeline de negocio en ambos frameworks

Vamos a construir columnas derivadas y luego dos agregaciones.


- la **lógica de análisis** es casi la misma,
- pero el **modelo de ejecución** cambia.

In [17]:
## 6.2 Definir funciones de enriquecimiento y análisis

def enriquecer_pandas(df):
    """Enriquecer DataFrame de Pandas con columnas derivadas"""
    df = df.copy()
    df['fecha'] = pd.to_datetime(df['fecha'])
    df['mes'] = df['fecha'].dt.month
    df['precio_unitario'] = df['ventas'] / df['unidades']
    df['venta_media'] = df['precio_unitario'] > df['precio_unitario'].median()
    return df

def enriquecer_dask(ddf):
    """Enriquecer DataFrame de Dask con columnas derivadas"""
    ddf = ddf.copy()
    ddf['fecha'] = dd.to_datetime(ddf['fecha'])
    ddf['mes'] = ddf['fecha'].dt.month
    ddf['precio_unitario'] = ddf['ventas'] / ddf['unidades']
    # ⚠️ En Dask, usar median_approximate() porque median() exacta no es práctica en distribuido
    ddf['venta_media'] = ddf['precio_unitario'] > ddf['precio_unitario'].median_approximate()
    return ddf

def resumen_ventas_pandas(df):
    """Calcular resumen de ventas en Pandas"""
    return {
        "ventas_por_region": df.groupby('region')['ventas'].sum().sort_values(ascending=False),
        "ventas_por_region_categoria": df.groupby(['region', 'categoria'])['ventas'].sum().sort_values(ascending=False),
        "promedio_precio_unitario": df.groupby('mes')['precio_unitario'].mean(),
    }

def resumen_ventas_dask(ddf):
    """Calcular resumen de ventas en Dask (retorna objetos lazy sin sort)"""
    return {
        "ventas_por_region": ddf.groupby('region')['ventas'].sum(),
        "ventas_por_region_categoria": ddf.groupby(['region', 'categoria'])['ventas'].sum(),
        "promedio_precio_unitario": ddf.groupby('mes')['precio_unitario'].mean(),
    }

print("✅ Funciones de análisis definidas")

✅ Funciones de análisis definidas


In [18]:
## 6.3 Benchmark: Pandas vs Dask

print("="*60)
print("BENCHMARK: Pandas vs Dask")
print("="*60)

# Benchmark con Pandas
print("\n🐼 Ejecutando con Pandas (secuencial)...")
inicio = perf_counter()
pdf_modelado = enriquecer_pandas(pdf)
resultado_pandas = resumen_ventas_pandas(pdf_modelado)
tiempo_pandas = perf_counter() - inicio

print(f"✅ Tiempo Pandas: {tiempo_pandas:.3f} s")
print(f"   Filas procesadas: {len(pdf):,}")
print(f"   Resultado (Top 5 regiones):")
display(resultado_pandas["ventas_por_region"].head())

BENCHMARK: Pandas vs Dask

🐼 Ejecutando con Pandas (secuencial)...
✅ Tiempo Pandas: 0.189 s
   Filas procesadas: 300,000
   Resultado (Top 5 regiones):


region
Norte        1.533238e+08
Sur          1.532104e+08
Occidente    1.530708e+08
Centro       1.526847e+08
Oriental     1.524881e+08
Name: ventas, dtype: float64

In [19]:
## 6.4 Benchmark: Dask (continuación)

# Benchmark con Dask
print("\n📦 Ejecutando con Dask (distribuido)...")
inicio = perf_counter()
ddf_modelado = enriquecer_dask(ddf)
# persist() cachea el resultado en memoria de workers
ddf_modelado = ddf_modelado.persist()
wait(ddf_modelado)  # Espera a que termine de cachear
resultado_dask_lazy = resumen_ventas_dask(ddf_modelado)
# compute() materializa todos los lazy objects
resultado_dask_computed = dask.compute(*resultado_dask_lazy.values())
tiempo_dask = perf_counter() - inicio

# Reconstruir diccionario con resultados materializados
resultado_dask = dict(zip(resultado_dask_lazy.keys(), resultado_dask_computed))
resultado_dask = {k: v.sort_values(ascending=False) if hasattr(v, "sort_values") else v for k, v in resultado_dask.items()}

print(f"✅ Tiempo Dask: {tiempo_dask:.3f} s")
print(f"   Particiones: {ddf.npartitions}")
print(f"   Resultado (Top 5 regiones):")
display(resultado_dask["ventas_por_region"].head())


📦 Ejecutando con Dask (distribuido)...
✅ Tiempo Dask: 0.626 s
   Particiones: 6
   Resultado (Top 5 regiones):


region
Norte        1.533238e+08
Sur          1.532104e+08
Occidente    1.530708e+08
Centro       1.526847e+08
Oriental     1.524881e+08
Name: ventas, dtype: float64

In [ ]:
# Benchmark con Dask
inicio = perf_counter()
ddf_modelado = enriquecer_dask(ddf).persist()
wait(ddf_modelado)
resultado_dask_lazy = resumen_ventas_dask(ddf_modelado)
resultado_dask = dask.compute(*resultado_dask_lazy.values())
tiempo_dask = perf_counter() - inicio

resultado_dask = dict(zip(resultado_dask_lazy.keys(), resultado_dask, strict=True))
resultado_dask = {k: v.sort_index() if hasattr(v, "sort_index") else v for k, v in resultado_dask.items()}
print(f"Tiempo Dask: {tiempo_dask:.2f} s")
display(resultado_dask["ventas_por_region"])

Tiempo Dask: 0.35 s


,ventas
region,
Centro,1.526847e+08
Norte,1.533238e+08
Occidente,1.530708e+08
Oriental,1.524881e+08
Sur,1.532104e+08


In [20]:
resultado_dask_computed

(region
 Centro       1.526847e+08
 Norte        1.533238e+08
 Occidente    1.530708e+08
 Oriental     1.524881e+08
 Sur          1.532104e+08
 Name: ventas, dtype: float64,
 region     categoria
 Centro     A            3.813522e+07
            B            3.851218e+07
            C            3.809642e+07
            D            3.794087e+07
 Norte      A            3.841058e+07
            B            3.820353e+07
            C            3.822887e+07
            D            3.848078e+07
 Occidente  A            3.843044e+07
            B            3.797916e+07
            C            3.779558e+07
            D            3.886563e+07
 Oriental   A            3.829187e+07
            B            3.839658e+07
            C            3.768812e+07
            D            3.811154e+07
 Sur        A            3.898422e+07
            B            3.796353e+07
            C            3.796895e+07
            D            3.829372e+07
 Name: ventas, dtype: float64,
 mes
 1     1

In [21]:
# Comparar resultados
print("\n" + "="*60)
print("VALIDACIÓN: Comparar resultados analíticos")
print("="*60)

try:
    # Validar que los resultados sean equivalentes
    # Nota: ventas_por_region es una Series, no un DataFrame
    pd_ventas = resultado_pandas["ventas_por_region"].sort_index()
    dask_ventas = resultado_dask["ventas_por_region"].sort_index()

    # Usar assert_series_equal para Series (no assert_frame_equal)
    pd.testing.assert_series_equal(
        pd_ventas,
        dask_ventas,
        check_exact=False,
        rtol=1e-5,
        atol=1e-5,
    )
    print("✅ Resultados validados: Pandas y Dask coinciden")
except AssertionError as e:
    print(f"⚠️ Diferencia detectada: {str(e)[:200]}")
except Exception as e:
    print(f"⚠️ Error en validación: {str(e)[:200]}")

# Tabla comparativa
print("\n" + "="*60)
print("RESUMEN: Tiempos de ejecución")
print("="*60)

comparacion = pd.DataFrame(
    {
        "Framework": ["Pandas", "Dask"],
        "Tiempo (s)": [tiempo_pandas, tiempo_dask],
        "Características": [
            f"Secuencial en 1 proceso",
            f"Distribuido en {ddf.npartitions} particiones",
        ],
    }
)

comparacion["Speedup"] = tiempo_pandas / comparacion["Tiempo (s)"]
comparacion["Más rápido"] = comparacion["Tiempo (s)"].apply(lambda x: "✅" if x == comparacion["Tiempo (s)"].min() else "❌")

display(comparacion)

print("\n📋 Interpretación:")
if tiempo_pandas < tiempo_dask:
    ratio = tiempo_dask / tiempo_pandas
    print(f"   Pandas es {ratio:.2f}x más rápido en este volumen.")
    print(f"   Razón: Dask tiene overhead de coordinación.")
    print(f"   Conclusión: Dask brilla con volúmenes mayores y paralelismo real.")
else:
    ratio = tiempo_pandas / tiempo_dask
    print(f"   Dask es {ratio:.2f}x más rápido en este volumen.")
    print(f"   Razón: Está aprovechando paralelismo distribuido.")


VALIDACIÓN: Comparar resultados analíticos
✅ Resultados validados: Pandas y Dask coinciden

RESUMEN: Tiempos de ejecución


,Framework,Tiempo (s),Características,Speedup,Más rápido
0,Pandas,0.188764,Secuencial en 1 proceso,1.000000,✅
1,Dask,0.625656,Distribuido en 6 particiones,0.301705,❌



📋 Interpretación:
   Pandas es 3.31x más rápido en este volumen.
   Razón: Dask tiene overhead de coordinación.
   Conclusión: Dask brilla con volúmenes mayores y paralelismo real.


### Cómo interpretar este resultado
- Dask tiene overhead.
- En datos pequeños o medianos, Pandas suele ganar.
- Dask empieza a brillar cuando aumentan los datos, las particiones o la necesidad de usar varios workers.

La pregunta correcta no es ¿Dask es mejor que Pandas? sino:

**¿este problema justifica paralelismo y/o distribución?**

## 8. Qué mirar en el dashboard de Dask

Mientras corre una tarea distribuida, abre el dashboard y observa:

- **Task Stream**: muestra si las tareas realmente se están repartiendo entre workers.
- **Workers**: cuántos workers hay, cuánta memoria usan y si alguno está saturado.
- **Graph**: deja ver la estructura del trabajo.
- **System / Memory**: útil para explicar que el cuello de botella no siempre es CPU.

Referencia oficial: la documentación de Dask describe el dashboard distribuido como una de las herramientas principales de diagnóstico.

## 9. Conectar este tema con tu clúster Docker del curso

Aquí está la parte importante para la sesión: **Colab no es el lugar ideal para demostrar cómputo distribuido real con tu Docker local**.

Entonces conviene separar los objetivos:

- **En Colab**: teoría, API, ejemplos pequeños y noción de particiones.
- **En local/Jupyter del curso**: conexión al scheduler real, varios workers y benchmark pesado.

Tu infraestructura relevante está en:

- `infraestructura/docker-compose.yml`
- `infraestructura/dask/jobs/benchmark_dask_vs_pandas.py`

El `docker-compose` monta `C:/Users/nib1l/Documents/diplocopia/BigData2026/infraestructura/dask/jobs:/app`, así que el benchmark puede ejecutarse directamente dentro del clúster.

usemos este script [benchmark_dask_vs_pandas](https://raw.githubusercontent.com/jazaineam1/BigData2026/refs/heads/prueba/infraestructura/dask/jobs/benchmark_dask_vs_pandas.py)

## 10. Comandos recomendados para la demostracion real con Docker

Ejecuta estos comandos desde la carpeta `infraestructura/`. En Colab quedan como referencia; la demostracion real debe hacerse en tu maquina local o en el Jupyter del entorno Docker.

```bash
cd infraestructura

docker compose --profile dask up -d --scale dask-worker=4 dask-scheduler dask-worker

docker compose ps

docker compose exec dask-scheduler python /app/benchmark_dask_vs_pandas.py --parts 6 --rows-per-part 1000000 --rebuild-data
```



```bash
docker compose exec dask-scheduler python /app/benchmark_dask_vs_pandas.py
  --parts 6 --rows-per-part 250000 --rebuild-data
```

## 11.1 Comandos reales en PowerShell para esta maquina

En este curso la ruta real fue:

```powershell
cd C:\Users\nib1l\Documents\diplocopia\BigData2026\infraestructura
docker compose --profile dask up -d dask-scheduler dask-worker --scale dask-worker=4
docker compose --profile dask ps
docker compose exec dask-scheduler python /app/benchmark_dask_vs_pandas.py --parts 6 --rows-per-part 1000 --rebuild-data
docker compose exec dask-scheduler python /app/benchmark_dask_vs_pandas.py --parts 6 --rows-per-part 250000 --rebuild-data
docker compose exec dask-scheduler python /app/benchmark_dask_vs_pandas.py --parts 6 --rows-per-part 500000 --rebuild-data
docker compose exec dask-scheduler python /app/benchmark_dask_vs_pandas.py --parts 24 --rows-per-part 500000 --rebuild-data
docker compose logs --tail=50 dask-scheduler
docker compose logs --tail=50 dask-worker
```

## 11.2 Resultados observados en el benchmark real

Estos tiempos salieron al ejecutar `benchmark_dask_vs_pandas.py` dentro del contenedor `dask-scheduler`, con validacion correcta entre Pandas y Dask.

| Partes x filas | Filas totales | Pandas | Dask | Lectura |
|---|---:|---:|---:|---|
| 6 x 1,000 | 6,000 | 0.13 s | 8.49 s | Pandas gana por mucho; Dask paga overhead |
| 6 x 50,000 | 300,000 | 1.09 s | 2.90 s | Todavia gana Pandas |
| 6 x 250,000 | 1,500,000 | 6.07 s | 8.49 s | La brecha se reduce, pero Pandas sigue ganando |
| 6 x 500,000 | 3,000,000 | 15.50 s | 11.92 s | Primera corrida donde Dask supero a Pandas |
| 24 x 500,000 | 12,000,000 | 71.39 s | 72.20 s | Quedan practicamente empatados |

### Lectura  importante

En esta maquina **Pandas gana claramente en datos pequenos y medianos**. El **primer caso medido** donde Dask fue mas rapido aparecio en `6 x 500,000 = 3,000,000` filas. Sin embargo, ese cruce **no fue perfectamente estable** al repetir con distinta configuracion de workers, asi que la conclusion correcta no es "a partir de X filas Dask siempre gana", sino:

- Dask empieza a competir cuando el volumen crece.
- El resultado depende del costo de coordinacion, del `shuffle`, del numero de particiones y de la carga real del cluster.
- En un benchmark con mucho `set_index()` y `shuffle`, el costo distribuido puede comerse parte del beneficio.


## 11.3 Que hacen los workers y de que depende tener mas

Un **worker** es un proceso que ejecuta tareas, usa CPU, usa memoria y guarda particiones temporales. Tener mas workers **no garantiza** mejor tiempo por si solo.

El beneficio de aumentar workers depende de:

- **Particiones suficientes**: si solo tienes 6 particiones, no siempre aprovechas bien 8 workers.
- **CPU real disponible**: si Docker Desktop o el equipo no tienen suficientes nucleos libres, los workers compiten entre si.
- **Memoria**: mas workers reparten memoria, pero tambien aumentan la coordinacion y el movimiento de datos.
- **Tipo de trabajo**: operaciones con `shuffle` fuerte, como `set_index()`, suelen costar bastante en distribuido.
- **I/O y red**: si el cuello de botella es leer, escribir o mover datos, agregar workers puede no ayudar mucho.
- **Overhead del scheduler**: mas workers significa mas coordinacion.

En otras palabras: mas workers ayudan cuando hay suficiente trabajo paralelo, suficientes particiones y recursos reales para sostenerlos.


## 11.4 Prueba real cambiando el numero de workers

Se repitio el mismo benchmark de `6 x 500,000 = 3,000,000` filas cambiando el numero de workers.

| Workers pedidos | Workers detectados por Dask | Pandas | Dask | Lectura |
|---:|---:|---:|---:|---|
| 1 | 1 | 9.33 s | 18.30 s | Con un solo worker Dask pierde casi todo el beneficio distribuido |
| 2 | 2 | 7.58 s | 11.34 s | Mejora frente a 1 worker, pero Pandas sigue ganando |
| 4 | 4 | 7.35 s | 11.18 s | En esta corrida 4 workers no bastaron para superar a Pandas |
| 8 | 5 detectados | 8.72 s | 12.92 s | Pedir mas contenedores no implico mas capacidad efectiva en este entorno |

### Que ensena esta prueba

- Mas workers no siempre reducen el tiempo linealmente.
- Si el dataset tiene pocas particiones respecto al numero de workers, parte del cluster queda subutilizado.
- En esta maquina, al pedir 8 workers, el scheduler solo vio 5 workers activos durante la prueba. Eso muestra que el limite real tambien depende del entorno Docker, de la memoria disponible y del arranque efectivo de los procesos.
- La pregunta correcta no es "cuantos workers mas puedo crear?", sino "mi carga tiene suficiente paralelismo y recursos para aprovecharlos?".


## 11.5 Si solo tienes un PC, cuando conviene Dask y cuando conviene Pandas

Con un solo PC, **Pandas** suele ser la mejor opcion cuando los datos caben bien en RAM y el analisis termina rapido. Tiene menos overhead, es mas simple de depurar y evita el costo de coordinacion entre procesos.

**Dask** empieza a tener sentido en un solo PC cuando los datos ya vienen en muchos archivos, cuando quieres procesar por particiones, cuando Pandas empieza a presionar memoria o cuando quieres preparar el pipeline para escalar despues a varios workers o a otra infraestructura.

La idea clave no es "Dask siempre es mas rapido", sino "Dask escala mejor cuando el problema deja de ser comodo para Pandas".


### Que debe ver el estudiante en esa ejecucion real

1. Pandas usa un solo proceso y concentra memoria en una sola maquina.
2. Dask detecta varios workers y reparte tareas.
3. En el dashboard aparecen tareas simultaneas en el **Task Stream**.
4. El benchmark valida que el resultado analitico sea equivalente en ambos casos.
5. Si el dataset crece, Dask puede empezar a superar a Pandas o, al menos, volverse mas sostenible en memoria.


## 12. ¿Cuándo usar Pandas y cuándo usar Dask?

| Situación | Mejor opción | Razón |
|---|---|---|
| Dataset pequeño y análisis rápido | Pandas | Menos overhead y depuración más simple |
| Datos medianos repartidos en muchos archivos | Dask | Lee y procesa por particiones |
| Necesitas paralelizar en varios workers | Dask | Aprovecha scheduler + workers |
| Todo cabe fácil en RAM y el flujo es simple | Pandas | Más directo |
| Quieres mostrar arquitectura de cómputo distribuido | Dask | Hace visible la coordinación entre nodos |
| ETL local grande antes de Spark | Dask | Buen puente entre Pandas y sistemas distribuidos mayores |

## 14. Ejercicios propuestos

1. Cambia el número de particiones y observa si mejora o empeora el tiempo.
2. Agrega una nueva métrica por `category` y compara tiempos nuevamente.
3. Ejecuta el benchmark con `2` y luego con `4` workers.
4. Toma una captura del dashboard y explica qué está ocurriendo.
5. Repite el experimento con un dataset más pequeño y redacta por qué Pandas gana allí.


## Referencias oficiales recomendadas

- Dask DataFrame Best Practices: https://docs.dask.org/en/stable/dataframe-best-practices.html
- Dask Distributed Diagnostics: https://docs.dask.org/en/stable/diagnostics-distributed.html
- Dask Delayed: https://docs.dask.org/en/stable/delayed.html